# WP5 — Base classifier and six baselines

**Authorship.** The methodological core of this notebook — the nested LOSO-CV loop with isotonic-calibrated LightGBM, the cost-sensitive Oracle global threshold, the cumulative-variance signal-quality heuristic, the four fixed-rule baselines, and the multi-strategy metrics — is the WP5 student's implementation, preserved with minimal edits for contract compliance. Comments are in Polish where the original student wrote them; do not strip them.

**What changed for contract compliance.** Three sections wrap the student's code:

1. **Inputs from the contract** (§1) — instead of a synthetic generator, read `synthetic/v1/probability_table.parquet` and `synthetic/v1/labels.parquet` per `docs/data_contract_v1.md`.
2. **Per-night ground truth** (§2 adapter) — the student's Oracle and signal-quality functions originally took *one label per patient*. The dataset has labels that change *within* a participant across the cycle (pre → post around ovulation). The functions are extended to use per-(participant, night) labels: `y_true_arr[decision_night]` rather than `y_true_patient`. Algorithm preserved; data shape generalised.
3. **Outputs to the contract** (§7) — write `decisions/<strategy_name>.parquet` matching `docs/pipeline_contract_v1.md` §5, plus rows in `tau_per_strategy.parquet` per §6.

**Two modes.** A `USE_REAL_DATA` flag at the top selects between (a) reading the synthetic probability table for fast iteration, or (b) running the LightGBM training pipeline on real mcPHASES features. The conference deliverable is mode (b); mode (a) is for debugging.

## 0. Setup

Imports follow the student's original choices: `lightgbm`, `LeaveOneGroupOut` for LOSO, `CalibratedClassifierCV` with isotonic regression for calibration.

In [ ]:
import sys, os
from pathlib import Path

# Walk up from cwd to find the repo root (works from any directory)
_p = Path().resolve()
while not (_p / 'utils' / 'dataset.py').is_file() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# biblioteki — preserved from the WP5 student's original notebook
import numpy as np
import pandas as pd
import time  # to do pomiaru czasu trwania obliczeń
from sklearn.model_selection import LeaveOneGroupOut  # to do algorytmu leave one subject out cross validation
from sklearn.metrics import accuracy_score, roc_auc_score  # do metryk

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

# Switch this to True to run the LightGBM training pipeline on real mcPHASES features.
USE_REAL_DATA = False

DATA = Path('synthetic/v1' if not USE_REAL_DATA else 'real/v1')
(DATA / 'decisions').mkdir(parents=True, exist_ok=True)
print(f'mode: {"REAL" if USE_REAL_DATA else "SYNTHETIC"}; pipeline root: {DATA}')

## 1. Inputs from the contract

When `USE_REAL_DATA = False`, read the synthetic probability table that `synthetic/generator.py` produced. The student's training loop is skipped in this mode — the probabilities already exist.

When `USE_REAL_DATA = True`, jump to §6 (Real-data training).

In [ ]:
prob_path = DATA / 'probability_table.parquet'
labels_path = DATA / 'labels.parquet'
DATA_LOADED = prob_path.is_file() and labels_path.is_file()

if DATA_LOADED:
    prob = pd.read_parquet(prob_path)
    lbls = pd.read_parquet(labels_path)
    pids = sorted(prob['participant_id'].unique())
    print(f'{len(prob):,} (participant, night) rows; {len(pids)} participants  (mode: {"REAL" if USE_REAL_DATA else "SYNTHETIC"})')

    # Adapter — convert contract dataframes to the per-patient list format
    # the student's functions expect: list-of-arrays, one entry per patient.
    y_prob_all = []   # per-patient probability arrays
    y_true_all = []   # per-patient label arrays (per-night, NOT per-patient — see header)
    all_x_patients = []   # per-patient feature arrays (for the heuristic; using probability as proxy)

    for pid in pids:
        g_prob = prob[prob['participant_id'] == pid].sort_values('night_index')
        g_lbls = lbls[lbls['participant_id'] == pid].sort_values('night_index')
        y_prob_all.append(g_prob['p_post_ovulatory'].to_numpy())
        y_true_all.append((g_lbls['binary_label'].to_numpy() == 'post').astype(int))
        all_x_patients.append(g_prob[['p_post_ovulatory']].to_numpy())

    print(f'adapter built: {len(y_prob_all)} patients ready for the student baselines')
else:
    print(f'No probability_table.parquet at {prob_path}.')
    print(f'  In synthetic mode, run synthetic/generator.py first.')
    print(f'  In real mode, §9 below trains the classifier and writes it.')

## 2. Oracle global threshold (student's implementation)

From the student's notebook, with two contract-compliance edits annotated inline. The cost-sensitive loss `cost_error * I[wrong] + cost_night * (k+1)` and the threshold-sweep architecture are unchanged.

In [ ]:
# GLOBAL ORACLE THRESHOLD — preserved from WP5 student's notebook
def find_oracle_threshold(y_prob_all, y_true_all, cost_error=1.0, cost_night=0.05):
    # ustalamy pierwotny najlepszy próg i najmniejsza stratę
    best_tau = 0.50
    min_loss = float('inf')  # nieskonczonosc zeby pierwszy policzony wynik stal sie tym najlepszym

    # CONTRACT EDIT: deterministic linspace instead of random sample (plan §9.3 reproducibility).
    thresholds_to_test = np.linspace(0.50, 0.99, 50)

    # dla kazdej testowanej wartosci tau
    for tau in thresholds_to_test:
        # ustawiamy obecna wartosc straty na 0
        current_loss = 0

        # dla pacjenta z grupy treningowej
        for y_prob, y_true_arr in zip(y_prob_all, y_true_all):
            decision_night = len(y_prob) - 1  # domyślnie bierzemy ostatnią noc
            # szukamy pierwszej nocy, w ktorej pewnosc modelu przekracza tau
            for i, p in enumerate(y_prob):
                # Jeśli szansa na chorobę >= tau LUB szansa na zdrowie (1-p) >= tau
                if p >= tau or p <= (1 - tau):
                    decision_night = i
                    break
            final_p = y_prob[decision_night]
            pred_class = 1 if final_p >= 0.5 else 0
            # CONTRACT EDIT: y_true is now a per-night array, not a per-patient scalar.
            # Index it at the decision night to compare against the actual phase at that moment.
            true_class = int(y_true_arr[decision_night]) if hasattr(y_true_arr, '__len__') else int(y_true_arr)
            # obliczamy strate — Obliczamy karę (stratę)
            error_penalty = cost_error if pred_class != true_class else 0
            time_penalty = (decision_night + 1) * cost_night

            current_loss += error_penalty + time_penalty

            # Jeśli ten próg wygenerował najmniejszą stratę, zapisujemy go
        if current_loss < min_loss:
            min_loss = current_loss
            best_tau = tau

    return best_tau


def baseline_oracle(y_prob, tau_star):
    """Aplikuje wyliczony próg tau* do pacjenta testowego."""
    decision_night = len(y_prob) - 1
    for i, p in enumerate(y_prob):
        if p >= tau_star or p <= (1 - tau_star):
            decision_night = i
            break
    # Zwracamy wynik z tej nocy, w której model poczuł się wystarczająco pewnie
    return y_prob[decision_night], decision_night

## 3. Signal-quality heuristic (student's implementation)

Decides at the first night where accumulated mean variance of the chosen feature column drops below a learned threshold. In synthetic mode, the "feature" used is the predicted probability itself (a placeholder); in real mode, it should be the nocturnal temperature variance per plan §3.5.

In [ ]:
# heurystyka — preserved from WP5 student's notebook
# musi mieć dostep do surowych danych -> dodajemy do argumentów funkcji x_train_val
def find_heuristic_threshold(x_train_val, y_prob_all, y_true_all, var_col_idx=0, cost_error=1.0, cost_night=0.05):
    # parametry poczatkowe tak jak w oracle
    best_th = 0.50
    min_loss = float('inf')

    # tesujemy progi wariancji (tu akurat 30 progow do przetestowania)
    thresholds_to_test = np.linspace(0.1, 0.9, 30)

    for th in thresholds_to_test:
        current_loss = 0

        # iterujemy przez pacjentów (mamy ich surowe X, predykcje i prawdziwą diagnozę) -> tak jak wyzej tylko wiecej o jeden parametr
        for x_pat, y_prob, y_true_arr in zip(x_train_val, y_prob_all, y_true_all):
            decision_night = len(y_prob) - 1

            # idziemy noc po nocy
            for i in range(len(y_prob)):
                # obliczamy ŚREDNIĄ wariancję od pierwszej nocy do obecnej (włącznie)
                mean_variance = np.mean(x_pat[:i+1, var_col_idx])  # x_pat to tabela danego pacjenta

                # jeśli sygnał jest dobrej jakości (wariancja spadła poniżej progu) -> PODEJMUJEMY DECYZJĘ
                if mean_variance < th:
                    decision_night = i
                    break

            # obliczanie straty - > identycznie jak w Oracle
            final_p = y_prob[decision_night]
            pred_class = 1 if final_p >= 0.5 else 0
            # CONTRACT EDIT: per-night label, see Oracle above
            true_class = int(y_true_arr[decision_night]) if hasattr(y_true_arr, '__len__') else int(y_true_arr)

            error_penalty = cost_error if pred_class != true_class else 0
            time_penalty = (decision_night + 1) * cost_night
            current_loss += error_penalty + time_penalty

        if current_loss < min_loss:
            min_loss = current_loss
            best_th = th

    return best_th


# aplikujemy wyuczony próg heurystyki do pacjenta
def baseline_heuristic(y_prob, x_patient, threshold, var_col_idx=0):
    decision_night = len(y_prob) - 1
    for i in range(len(y_prob)):
        mean_variance = np.mean(x_patient[:i+1, var_col_idx])
        if mean_variance < threshold:
            decision_night = i
            break
    return y_prob[decision_night], decision_night

## 4. Always-predict and Fixed-k (student's implementation)

In [ ]:
# linia bazowa always predict — preserved from WP5 student's notebook
def baseline_always_predict(y_prob_all):
    return y_prob_all[0], 0


# linie bazowe Fixed-3/5/7 — preserved from WP5 student's notebook
def baseline_fixed_k(y_prob_all, k):
    if len(y_prob_all) >= k:
        decision_night = k - 1
    else:
        decision_night = len(y_prob_all) - 1
    return y_prob_all[decision_night], decision_night

## 5. Apply all six baselines and compute thresholds (student's implementation)

Per-fold Oracle and signal-quality threshold learning, with the timing checkpoint and the 24-hour walltime guard from plan §3.6.

In **synthetic mode** the LightGBM training is skipped: the probability table already exists from `synthetic/generator.py`, so we just learn the Oracle/heuristic thresholds via leave-one-subject-out on the existing probabilities. In **real mode** §6 trains LightGBM with isotonic calibration and produces the probability table from scratch.

In [ ]:
if DATA_LOADED:
    # Synthetic-mode adapter for the student's per-fold logic.
    # In real mode this is replaced by §6's nested LOSO + LightGBM loop.
    all_oracle_taus = []
    all_heuristic_thresholds = []
    start_time_total = time.time()

    for held_out_idx, _ in enumerate(pids):
        # train_val = everyone except held_out_idx
        idx_others = [i for i in range(len(pids)) if i != held_out_idx]
        y_prob_train = [y_prob_all[i] for i in idx_others]
        y_true_train = [y_true_all[i] for i in idx_others]
        x_train = [all_x_patients[i] for i in idx_others]

        tau_star = find_oracle_threshold(y_prob_train, y_true_train)
        heur_th = find_heuristic_threshold(x_train, y_prob_train, y_true_train)

        all_oracle_taus.append(tau_star)
        all_heuristic_thresholds.append(heur_th)

    total_time = time.time() - start_time_total
    print(f'learned Oracle and heuristic thresholds across {len(pids)} LOSO folds in {total_time:.1f} s')
    print(f'  Oracle τ*: median {np.median(all_oracle_taus):.3f}, range [{min(all_oracle_taus):.3f}, {max(all_oracle_taus):.3f}]')
    print(f'  Heuristic threshold: median {np.median(all_heuristic_thresholds):.3f}, range [{min(all_heuristic_thresholds):.3f}, {max(all_heuristic_thresholds):.3f}]')

In [ ]:
# Apply baselines & metrics — runs in synthetic mode automatically.
# In real mode, §9 trains the classifier then runs the same logic itself.
if DATA_LOADED:
    # wyniki z linii bazowych — preserved from WP5 student's notebook (per-patient apply loop)
    results_always_predict = []
    results_fixed_3 = []
    results_fixed_5 = []
    results_fixed_7 = []
    results_oracle = []
    results_heuristic = []

    max_nights_per_patient = [len(p) for p in y_prob_all]

    # dla kazdego pacjenta
    for i in range(len(y_prob_all)):
        # 1. wyciągamy dane z odpowiednich pudełek po numerze (i)
        y_prob_pat = y_prob_all[i]
        tau_star = all_oracle_taus[i]
        x_pat = all_x_patients[i]
        heur_th = all_heuristic_thresholds[i]

        # 2. aplikujemy pierwsze 5 linii bazowych
        pred_always = baseline_always_predict(y_prob_pat)
        pred_fix_3 = baseline_fixed_k(y_prob_pat, k=3)
        pred_fix_5 = baseline_fixed_k(y_prob_pat, k=5)
        pred_fix_7 = baseline_fixed_k(y_prob_pat, k=7)
        pred_oracle = baseline_oracle(y_prob_pat, tau_star)

        # 3. aplikujemy Heurystykę - podajemy zmienne W SZTYWNEJ KOLEJNOŚCI!
        # (Predykcje -> Tabela Pacjenta -> Wyliczony Próg)
        pred_heur = baseline_heuristic(y_prob_pat, x_pat, heur_th, var_col_idx=0)

        # 4. zapisujemy wyniki
        results_always_predict.append(pred_always)
        results_fixed_3.append(pred_fix_3)
        results_fixed_5.append(pred_fix_5)
        results_fixed_7.append(pred_fix_7)
        results_oracle.append(pred_oracle)
        results_heuristic.append(pred_heur)

    # Podgląd dla pacjenta nr 1
    print(f'Wyniki dla Pacjenta nr 1 ({pids[0]}):')
    print(f'  Oceny modelu (pierwsze 5 nocy): {np.round(y_prob_all[0][:5], 2)}')
    print(f'  Decyzja Always-Predict:   {results_always_predict[0][0]:.2f}  (noc {results_always_predict[0][1]+1})')
    print(f'  Decyzja Fixed-3:          {results_fixed_3[0][0]:.2f}  (noc {results_fixed_3[0][1]+1})')
    print(f'  Wyznaczony próg Oracle:   {all_oracle_taus[0]:.2f}')
    print(f'  Decyzja Oracle:           {results_oracle[0][0]:.2f}  (noc {results_oracle[0][1]+1})')
    print(f'  Wyznaczony próg Heuryst.: {all_heuristic_thresholds[0]:.3f}')
    print(f'  Decyzja Heurystyki:       {results_heuristic[0][0]:.2f}  (noc {results_heuristic[0][1]+1})')

## 6. Metrics across all baselines (student's implementation)

Accuracy, ROC AUC, selective accuracy, deferral rate, median τᵢ, IQR τᵢ — covering plan §3.7.

In [ ]:
# metryki — preserved from WP5 student's notebook with one extension for per-night y_true
def calculate_all_metrics(y_true_per_patient, baseline_results, max_nights):
    """y_true_per_patient: list-of-arrays (per-patient per-night labels) OR list of scalars.
       baseline_results: list of (prob, decision_night_idx) tuples from one baseline.
    """
    # wyciągamy prawdopodobieństwa i decyzje (zaokrąglamy ułamki do 0 lub 1)
    probs = np.array([res[0] for res in baseline_results])
    preds = (probs >= 0.5).astype(int)

    # CONTRACT EDIT: derive scalar y_true per patient as the label at that patient's decision night.
    truths = []
    for arr_or_scalar, (_, dn) in zip(y_true_per_patient, baseline_results):
        if hasattr(arr_or_scalar, '__len__'):
            truths.append(int(arr_or_scalar[dn]))
        else:
            truths.append(int(arr_or_scalar))
    truths = np.array(truths)

    # wyciągamy czasy decyzji (\tau_i) - dodajemy 1
    decision_nights = np.array([res[1] for res in baseline_results]) + 1
    max_n = np.array(max_nights)

    # 1. Overall Accuracy
    acc = accuracy_score(truths, preds)
    try:
        roc_auc = roc_auc_score(truths, probs)
    except ValueError:
        roc_auc = np.nan

    # 2. Deferral rate — odroczenie oznacza, że system nie podjął decyzji i dotrwał do ostatniej dostępnej nocy
    deferred_mask = (decision_nights == max_n)
    deferral_rate = float(np.mean(deferred_mask))

    # 3. Selective accuracy
    if np.sum(~deferred_mask) > 0:
        selective_acc = accuracy_score(truths[~deferred_mask], preds[~deferred_mask])
    else:
        selective_acc = np.nan

    # 4. IQR of \tau_i (Evidence heterogeneity)
    q75, q25 = np.percentile(decision_nights, [75, 25])
    iqr_tau = float(q75 - q25)

    return {
        'Accuracy (%)': round(acc * 100, 2),
        'ROC AUC': round(roc_auc, 3) if not np.isnan(roc_auc) else 'N/A',
        'Selective Acc (%)': round(selective_acc * 100, 2) if not np.isnan(selective_acc) else 'N/A',
        'Deferral Rate (%)': round(deferral_rate * 100, 2),
        'Median tau_i (nights)': float(np.median(decision_nights)),
        'IQR of tau_i (nights)': round(iqr_tau, 2),
    }


if DATA_LOADED:
    # Tworzymy słownik ze wszystkimi wynikami
    all_results = {
        'always_predict': results_always_predict,
        'fixed_3': results_fixed_3,
        'fixed_5': results_fixed_5,
        'fixed_7': results_fixed_7,
        'oracle_global': results_oracle,
        'signal_quality': results_heuristic,
    }

    metrics_summary = []
    for name, res in all_results.items():
        metrics = calculate_all_metrics(y_true_all, res, max_nights_per_patient)
        metrics['Strategy'] = name
        metrics_summary.append(metrics)

    df_metrics = pd.DataFrame(metrics_summary)
    cols = ['Strategy', 'Accuracy (%)', 'ROC AUC', 'Selective Acc (%)', 'Deferral Rate (%)', 'Median tau_i (nights)', 'IQR of tau_i (nights)']
    df_metrics = df_metrics[cols]
    print('wyniki — six baselines side by side:')
    df_metrics

## 7. Outputs to the contract

Convert the in-memory `results_*` lists into one parquet file per strategy under `decisions/<strategy_name>.parquet`, all sharing the universal schema (`docs/pipeline_contract_v1.md` §5). Plus τᵢ entries appended to `tau_per_strategy.parquet` per §6.

This is the only section the student didn't write — it's the contract glue.

In [ ]:
# Apply baselines & metrics — runs in synthetic mode automatically.
# In real mode, §9 trains the classifier then runs the same logic itself.
if DATA_LOADED:
    STRATEGY_FROM_RESULTS = {
        'always_predict': results_always_predict,
        'fixed_3': results_fixed_3,
        'fixed_5': results_fixed_5,
        'fixed_7': results_fixed_7,
        'oracle_global': results_oracle,
        'signal_quality': results_heuristic,
    }

    tau_rows = []
    for strat_name, results in STRATEGY_FROM_RESULTS.items():
        rows = []
        for pid, (prob_at_decision, decision_night_idx), full_prob_array in zip(pids, results, y_prob_all):
            n_nights = len(full_prob_array)
            decision_night = decision_night_idx + 1  # 1-indexed in the contract
            for night_idx in range(1, n_nights + 1):
                if night_idx < decision_night:
                    decision = 'defer'
                    pred_label = None
                elif night_idx == decision_night:
                    decision = 'predict'
                    pred_label = 'post' if prob_at_decision >= 0.5 else 'pre'
                else:
                    # Fixed-k commits at k and stops emitting rows for later nights;
                    # always_predict / oracle / heuristic continue at the same decision.
                    if strat_name.startswith('fixed_'):
                        break
                    decision = 'predict'
                    pred_label = 'post' if prob_at_decision >= 0.5 else 'pre'
                rows.append({
                    'participant_id': pid,
                    'night_index': int(night_idx),
                    'cumulative_k': int(night_idx),
                    'strategy_name': strat_name,
                    'decision': decision,
                    'predicted_label': pred_label,
                    'prediction_set_size': pd.NA,
                    'mondrian_stratum': pd.NA,
                    'is_synthetic': not USE_REAL_DATA,
                })
        df = pd.DataFrame(rows).astype({
            'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
            'strategy_name': 'string', 'decision': 'string',
            'predicted_label': 'string', 'prediction_set_size': 'Int8',
            'mondrian_stratum': 'string', 'is_synthetic': 'bool',
        })
        df.to_parquet(DATA / f'decisions/{strat_name}.parquet', index=False)
        n_pred = (df['decision'] == 'predict').sum()
        print(f'  decisions/{strat_name}.parquet  →  {len(df):>5} rows, {n_pred:>4} predict')

        # τᵢ row per (patient, strategy) — value = decision_night (1-indexed)
        for pid, (_, decision_night_idx) in zip(pids, results):
            tau_rows.append({
                'participant_id': pid,
                'strategy_name': strat_name,
                'tau_i': int(decision_night_idx + 1),
                'convergence_status': 'converged',
                'is_synthetic': not USE_REAL_DATA,
            })

    tau_new = pd.DataFrame(tau_rows).astype({
        'participant_id': 'string', 'strategy_name': 'string',
        'tau_i': 'Int32', 'convergence_status': 'string', 'is_synthetic': 'bool',
    })
    tau_path = DATA / 'tau_per_strategy.parquet'
    if tau_path.exists():
        existing = pd.read_parquet(tau_path)
        existing = existing[~existing['strategy_name'].isin(STRATEGY_FROM_RESULTS.keys())]
        combined = pd.concat([existing, tau_new], ignore_index=True)
    else:
        combined = tau_new
    combined.to_parquet(tau_path, index=False)
    print(f"tau_per_strategy.parquet — {len(combined)} rows, strategies: {sorted(combined['strategy_name'].unique())}")

## 8. Performance vs. nights (student's plot)

Sweep Fixed-k for k from 1 to max nights, with Oracle and signal-quality as horizontal references.

In [ ]:
# Apply baselines & metrics — runs in synthetic mode automatically.
# In real mode, §9 trains the classifier then runs the same logic itself.
if DATA_LOADED:
    import matplotlib.pyplot as plt

    max_possible_nights = max(len(p) for p in y_prob_all)
    k_values = list(range(1, max_possible_nights + 1))

    fixed_k_roc_auc = []
    fixed_k_accuracy = []

    for k in k_values:
        preds_for_k = []
        truths_for_k = []
        for y_prob_pat, y_true_arr in zip(y_prob_all, y_true_all):
            prob, dn = baseline_fixed_k(y_prob_pat, k)
            preds_for_k.append(prob)
            truths_for_k.append(int(y_true_arr[dn]) if hasattr(y_true_arr, '__len__') else int(y_true_arr))
        probs_array = np.array(preds_for_k)
        binary_preds = (probs_array >= 0.5).astype(int)
        truths_array = np.array(truths_for_k)
        try:
            fixed_k_roc_auc.append(roc_auc_score(truths_array, probs_array))
        except ValueError:
            fixed_k_roc_auc.append(np.nan)
        fixed_k_accuracy.append(accuracy_score(truths_array, binary_preds))

    oracle_auc = df_metrics.loc[df_metrics['Strategy'] == 'oracle_global', 'ROC AUC'].iloc[0]
    heuristic_auc = df_metrics.loc[df_metrics['Strategy'] == 'signal_quality', 'ROC AUC'].iloc[0]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(k_values, fixed_k_roc_auc, marker='o', linewidth=2, markersize=8,
            color='#1f77b4', label='Fixed-k (decyduj dokładnie po k nocach)')
    if oracle_auc != 'N/A':
        ax.axhline(y=float(oracle_auc), color='#2ca02c', linestyle='--', linewidth=2,
                   label=f'Oracle global (AUC: {oracle_auc})')
    if heuristic_auc != 'N/A':
        ax.axhline(y=float(heuristic_auc), color='#d62728', linestyle='--', linewidth=2,
                   label=f'Signal-quality (AUC: {heuristic_auc})')
    ax.set_title('Performance vs. Nights (ROC AUC)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of nights (k)', fontsize=12)
    ax.set_ylabel('ROC AUC', fontsize=12)
    ax.set_xticks(k_values)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    Path('docs/figures').mkdir(parents=True, exist_ok=True)
    plt.savefig('docs/figures/wp5_performance_vs_nights.png', dpi=150)
    plt.show()
    print('saved docs/figures/wp5_performance_vs_nights.png')

## 9. Real-data training (student's nested LOSO + LightGBM)

When `USE_REAL_DATA = True`, this section trains LightGBM with isotonic calibration under nested LOSO-CV, exactly as the student wrote it. The walltime check at fold 1 (plan §3.6) breaks out if the projected total exceeds 24 hours.

Inputs come from `utils.dataset.load_mcphases()` after feature extraction; the output is `probability_table.parquet` matching the contract.

In [ ]:
if USE_REAL_DATA and not DATA_LOADED:
    # No real probability_table yet — need to train.
    print('Real-data training pipeline runs here.')
    try:
        import lightgbm as lgb
    except (ImportError, OSError) as exc:
        raise RuntimeError(
            'lightgbm could not load. On macOS run `brew install libomp` first; '
            'on Colab libomp is preinstalled. Underlying error: ' + str(exc)
        ) from exc
    from sklearn.calibration import CalibratedClassifierCV
    from utils.dataset import setup as _setup, load_mcphases as _load
    _setup()
    data = _load()

    # TODO (real-data): build per-(participant, night) feature matrix from `data`.
    # Recommended starter set:
    #   - nightly mean and std of skin temperature (from data['computed_temperature'])
    #   - mean HR per night (aggregate data.load('heart_rate', participant_id=...) to nightly)
    #   - mean HRV rmssd per night (same pattern)
    #   - mean glucose per night (data.load('glucose', ...))
    # Filter feature rows to valid nights only by joining on
    # `pd.read_parquet(DATA / 'preprocessing/valid_nights.parquet')`.
    raise NotImplementedError(
        'Real-data feature extraction TODO — fill in this cell.\n'
        'See _internal/notebooks/03_wp5_baselines_old_starter.ipynb for hints, '
        'and the WP5 student\'s nested-LOSO loop above for the training scaffold.'
    )

    # Once x, y, groups arrays exist, the student's nested-LOSO loop applies verbatim:
    #   logo = LeaveOneGroupOut()
    #   for train_val_index, test_index in logo.split(x, y, groups):
    #       ...
    #   After the loop, populate y_prob_all, y_true_all, all_x_patients, pids
    #   and write probability_table.parquet matching the contract.

elif USE_REAL_DATA and DATA_LOADED:
    print('Real probability_table.parquet already on disk; baselines applied above. '
          'Set DATA_LOADED=False to retrain.')
else:
    # synthetic mode — already handled
    pass

## What you've built

Six baselines, all with the same metric framework, all writing to the same contract-compliant decision file format. The cost-sensitive Oracle and the cumulative-variance signal-quality heuristic are your novel implementations on top of the standard fixed-k and always-predict patterns.

## Next

- Set `USE_REAL_DATA = True` and complete the feature-extraction TODO in §9.
- Send `decisions/oracle_global.parquet` and `decisions/signal_quality.parquet` to the WP6 student so her covariate-conditional comparison table includes your numbers.
- The WP7 student will pick up `decisions/*.parquet` files automatically — no further coordination needed.